In [3]:
import truststore
truststore.inject_into_ssl()

In [4]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
import bs4
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.prompts import ChatPromptTemplate

#### INDEXING ####

# Load Documents
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
docs = loader.load()

# Split
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)

# Embed (free - runs locally)
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

#### RETRIEVAL and GENERATION ####

# Prompt
prompt = ChatPromptTemplate.from_template("""Answer the question based only on the following context:
{context}
Question: {question}
""")

# LLM
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt

    | llm
    | StrOutputParser()
)


# Question
print(rag_chain.invoke("How are you"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8461.39it/s]


I'm functioning within the parameters set by my resources and performance evaluation guidelines. I have access to internet searches, long-term memory management, GPT-3.5 powered agents, and file output. I continuously review and analyze my actions to ensure optimal performance, and I strive to complete tasks efficiently in the least number of steps.


Indexing

In [11]:
question = "What kinds of pets do I like?"
document = "My favorite pet is a cat."

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
embd = HuggingFaceEmbeddings()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6660.18it/s]


In [18]:
query_result = embd.embed_query(question)
document_result = embd.embed_query(document)
len(query_result)

768

In [21]:
import numpy as np

def cosine_similarity(vec1, vec2):
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    return dot_product / (norm_vec1 * norm_vec2)

similarity = cosine_similarity(query_result, document_result)
print("Cosine Similarity:", similarity)

Cosine Similarity: 0.5595268089529775


In [22]:
import bs4
from langchain_community.document_loaders import WebBaseLoader
loader = WebBaseLoader(
    web_paths=("https://lilianweng.github.io/posts/2023-06-23-agent/",),
    bs_kwargs=dict(
        parse_only=bs4.SoupStrainer(
            class_=("post-content", "post-title", "post-header")
        )
    ),
)
blog_docs = loader.load()

In [23]:
# class_=("post-content", "post-title", "post-header")

# basically bta rha ha kya kya pick karna ha and kya skip, for eg.
# ├── <nav class="navbar">navbar</nav>          skip
# ├── <div class="post-header">post-header</div>     le lo
# ├── <div class="post-title">post-title</div>      le lo
# ├── <div class="post-content">post-content</div>    le lo
# └── <footer>...</footer>                   skip
# fayda — poora HTML parse nahi karta, sirf relevant parts — faster & cleaner.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)

# Make splits
splits = text_splitter.split_documents(blog_docs)

In [38]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)
retriever = vectorstore.as_retriever()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7133.99it/s]


In [42]:
type(vectorstore._collection)

chromadb.api.models.Collection.Collection

In [43]:
import chromadb

client = chromadb.Client()

collection = client.create_collection("my_docs")

collection.add(
    documents=["I love AI"],
    ids=["1"]
)

results = collection.query(
    query_texts=["machine learning"],
    n_results=1
)

print(results)

/Users/bharat.goyal1/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [01:06<00:00, 1.25MiB/s]


{'ids': [['1']], 'embeddings': None, 'documents': [['I love AI']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None]], 'distances': [[1.3316326141357422]]}


Retrieval

In [47]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
vectorstore = Chroma.from_documents(documents=splits, 
                                    embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2"))


retriever = vectorstore.as_retriever(search_kwargs={"k": 1})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9726.96it/s]


In [53]:
docs = retriever.invoke("What is Task Decomposition?")
docs

[Document(metadata={'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/'}, page_content='Task decomposition can be done (1) by LLM with simple prompting like "Steps for XYZ.\\n1.", "What are the subgoals for achieving XYZ?", (2) by using task-specific instructions; e.g. "Write a story outline." for writing a novel, or (3) with human inputs.\nAnother quite distinct approach, LLM+P (Liu et al. 2023), involves relying on an external classical planner to do long-horizon planning. This approach utilizes the Planning Domain Definition Language (PDDL) as an intermediate interface to describe the planning problem. In this process, LLM (1) translates the problem into “Problem PDDL”, then (2) requests a classical planner to generate a PDDL plan based on an existing “Domain PDDL”, and finally (3) translates the PDDL plan back into natural language. Essentially, the planning step is outsourced to an external tool, assuming the availability of domain-specific PDDL and a suitable planner

Generation

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

# Prompt
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n'), additional_kwargs={})])

In [56]:
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

In [57]:
chain = prompt | llm

In [58]:
chain.invoke({"context":docs,"question":"What is Task Decomposition?"})

AIMessage(content='According to the provided context, Task Decomposition can be done in the following ways:\n\n1. By using a Large Language Model (LLM) with simple prompting, such as asking for steps or subgoals to achieve a task.\n2. By using task-specific instructions, like writing a story outline for writing a novel.\n3. With human inputs.\n\nAdditionally, another approach called LLM+P involves using an external classical planner to do long-horizon planning, where the LLM translates the problem into a Planning Domain Definition Language (PDDL) and then requests a plan from the planner. However, the context does not provide a general definition of Task Decomposition beyond these methods.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 138, 'prompt_tokens': 306, 'total_tokens': 444, 'completion_time': 0.380645558, 'completion_tokens_details': None, 'prompt_time': 0.017159796, 'prompt_tokens_details': None, 'queue_time': 0.162414604, 'total_time': 0.3978

In [60]:

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

rag_chain.invoke("What is Task Decomposition?")

'According to the provided context, Task Decomposition can be done in the following ways:\n\n1. By using a Large Language Model (LLM) with simple prompting, such as asking for steps or subgoals to achieve a task.\n2. By using task-specific instructions, like writing a story outline for writing a novel.\n3. With human inputs.\n\nAdditionally, another approach, LLM+P, involves using an external classical planner to do long-horizon planning, where the LLM translates the problem into a Planning Domain Definition Language (PDDL) and then requests a plan from the planner. However, the context does not provide a direct definition of Task Decomposition, but rather describes methods for achieving it.'